# AAPD: qual método recupera melhor as classes originais com 54 tópicos?

Comparação entre **Híbrido (LLDA + LDA)**, sobre títulos já gerados pelo LLDA, e
**LDA direto**, sobre os resumos dos mesmos 2.000 artigos. Ambos usam **n = K = 54**,
igual ao número de classes originais. O notebook anterior com K=10 permanece intacto.

O foco é a fidelidade às **classes multi-rótulo**, e não apenas separação geométrica.
Ter 54 tópicos não garante correspondência com as 54 classes: tópicos têm índices
arbitrários e classes podem coocorrer. Avaliamos duas perguntas:

1. **Correspondência direta:** após alinhar tópicos e classes no treino, qual método
   coloca as classes verdadeiras nas primeiras posições? **LRAP alinhado** é o
   critério principal, acompanhado de ranking loss, cobertura e acerto top-1.
2. **Informação recuperável:** uma mesma sonda linear consegue recuperar os rótulos
   a partir dos pesos dos tópicos? Medimos LRAP, F1 micro/macro e Jaccard.

Usamos um holdout fixo de 25%. Vocabulários, LDA, alinhamento e sondas são ajustados
**somente no treino**. Os rótulos de teste são usados apenas para avaliação.
Os títulos LLDA são reutilizados do CSV; este notebook não chama Ollama.

In [ ]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    label_ranking_average_precision_score, label_ranking_loss, coverage_error,
    f1_score, jaccard_score, normalized_mutual_info_score, adjusted_rand_score,
    precision_recall_fscore_support,
)
import plotly.express as px

N_TOPICS = 54
SEED = 42
MAX_ITER = 35
TEST_SIZE = 0.25
THRESHOLD = 0.3  # fixo para ambos; não escolhido usando o teste
N_JOBS = 2
pd.set_option('display.max_columns', None)

# Suporta iniciar o kernel na raiz do projeto ou na pasta do notebook.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'dataset_loader.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Inicie o kernel na raiz do projeto ou na pasta do notebook.')
sys.path.insert(0, str(ROOT))
from dataset_loader import AAPD_ARXIV_CATEGORIES

DATA_DIR = ROOT / 'experimento_aapd_comparativo'
df = pd.read_csv(DATA_DIR / 'experimento_aapd_comparativo.csv').reset_index(drop=True)
required = ['doc_id', 'classes_reais_aapd', 'topico_llda', 'resumo_texto']
if df[required].isna().any().any() or df['doc_id'].duplicated().any():
    raise ValueError('Há campos ausentes ou doc_id duplicado no CSV.')
all_classes = list(AAPD_ARXIV_CATEGORIES)
assert len(all_classes) == N_TOPICS == len(set(all_classes))
class_index = {c: i for i, c in enumerate(all_classes)}
labels = [list(dict.fromkeys(c.strip() for c in row.split(',') if c.strip()))
          for row in df['classes_reais_aapd']]
unknown = {c for row in labels for c in row} - set(all_classes)
if unknown or any(not row for row in labels):
    raise ValueError(f'Rótulos vazios ou classes desconhecidas: {unknown}')
Y = np.zeros((len(df), N_TOPICS), dtype=int)
for i, row in enumerate(labels):
    Y[i, [class_index[c] for c in row]] = 1
primary = np.array([row[0] for row in labels])
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, random_state=SEED)
Y_train, Y_test = Y[train_idx], Y[test_idx]
class_support = pd.DataFrame({'Classe': all_classes, 'Total': Y.sum(0),
                              'Treino': Y_train.sum(0), 'Teste': Y_test.sum(0)})
print(f'{len(df)} documentos | {N_TOPICS} classes | treino={len(train_idx)} | teste={len(test_idx)}')
print(f'Média de rótulos por documento: {Y.sum(1).mean():.2f}')
display(class_support)

## Treinamento comparável com K=54

Mesma semente, número de tópicos, iterações e stopwords. Mantemos os vetorizadores
do experimento original: unigramas/bigramas nos títulos e unigramas nos resumos,
com limites de frequência diferentes. Assim, comparamos os dois pipelines completos,
e não isolamos o efeito do LLDA. Cada método aprende seu próprio vocabulário no treino.

In [ ]:
academic_stops = {
    'algorithm', 'algorithms', 'method', 'methods', 'problem', 'problems',
    'approach', 'approaches', 'analysis', 'paper', 'study', 'technique',
    'techniques', 'based', 'using', 'proposed', 'new', 'framework', 'system', 'systems'
}
stop_words = sorted(text.ENGLISH_STOP_WORDS.union(academic_stops))
def clean_topic(value):
    return re.sub(r'\s+', ' ', value.replace('\n', ' ').strip("\"'`")).strip()

inputs = {
    'Híbrido (LLDA + LDA)': (
        df['topico_llda'].map(clean_topic).to_numpy(),
        CountVectorizer(stop_words=stop_words, min_df=2, ngram_range=(1, 2),
                        token_pattern=r'(?u)\b[a-zA-Z]{3,}\b')),
    'LDA direto': (
        df['resumo_texto'].to_numpy(),
        CountVectorizer(stop_words=stop_words, min_df=3, max_df=0.85, max_features=10000,
                        token_pattern=r'(?u)\b[a-zA-Z]{3,}\b')),
}
models = {}
for name, (documents, vectorizer) in inputs.items():
    dtm_train = vectorizer.fit_transform(documents[train_idx])
    dtm_test = vectorizer.transform(documents[test_idx])
    lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=SEED,
                                   max_iter=MAX_ITER, learning_method='batch', n_jobs=N_JOBS)
    theta_train = lda.fit_transform(dtm_train)
    theta_test = lda.transform(dtm_test)
    models[name] = dict(vectorizer=vectorizer, lda=lda, train=theta_train, test=theta_test)
    print(f'{name}: {theta_train.shape} treino / {theta_test.shape} teste; '
          f'{dtm_train.shape[1]} termos; '
          f'{np.count_nonzero(dtm_test.getnnz(axis=1) == 0)} documentos de teste sem termos no vocabulário')

## Alinhamento de 54 tópicos com 54 classes

Calculamos a massa média de cada tópico nos documentos positivos de cada classe,
normalizando pelo suporte da classe no treino. O algoritmo Húngaro encontra a
permutação que maximiza a soma dessas associações, permitindo um tópico por classe.
Classes sem positivos no treino recebem associação zero e não têm alinhamento aprendido.

Essa correspondência é uma hipótese restritiva: classes que sempre coocorrem
(como podem ocorrer pares de categorias do arXiv) não precisam ocupar tópicos distintos.
Por isso também avaliamos a sonda linear, que permite recuperar várias classes de um tópico.

In [ ]:
for name, model in models.items():
    association = (model['train'].T @ Y_train) / np.maximum(Y_train.sum(0), 1)[None, :]
    topics, classes = linear_sum_assignment(-association)
    topic_for_class = np.empty(N_TOPICS, dtype=int)
    topic_for_class[classes] = topics
    model['topic_for_class'] = topic_for_class
    model['aligned'] = model['test'][:, topic_for_class]
    words = model['vectorizer'].get_feature_names_out()
    model['alignment'] = pd.DataFrame({
        'Classe': all_classes,
        'Tópico': topic_for_class + 1,
        'Associação no treino': association[topic_for_class, np.arange(N_TOPICS)],
        'Palavras-chave': [', '.join(words[model['lda'].components_[t].argsort()[-5:][::-1]])
                          for t in topic_for_class],
    })
    print(name)
    display(model['alignment'])

## Comparação principal: proximidade direta das classes originais

LRAP e ranking loss consideram todas as classes verdadeiras. Coverage error mede
quantas **classes** precisam ser percorridas no ranking para cobrir os rótulos reais.
Acerto top-1 aceita qualquer classe verdadeira do documento. NMI e ARI usam apenas
a primeira classe do CSV, como diagnósticos secundários; não representam todo o alvo multi-rótulo.

In [ ]:
def ranking_metrics(y_true, scores):
    return {
        'LRAP': label_ranking_average_precision_score(y_true, scores),
        'Ranking loss': label_ranking_loss(y_true, scores),
        'Coverage error': coverage_error(y_true, scores),
        'Acerto top-1': y_true[np.arange(len(y_true)), scores.argmax(1)].mean(),
    }
rows = []
for name, model in models.items():
    metrics = ranking_metrics(Y_test, model['aligned'])
    metrics['NMI (classe primária)'] = normalized_mutual_info_score(primary[test_idx], model['test'].argmax(1))
    metrics['ARI (classe primária)'] = adjusted_rand_score(primary[test_idx], model['test'].argmax(1))
    rows.append({'Método': name, **metrics})
direct_results = pd.DataFrame(rows).set_index('Método')
display(direct_results.round(4))

higher_is_better = {'LRAP': True, 'Ranking loss': False, 'Coverage error': False,
                    'Acerto top-1': True, 'NMI (classe primária)': True, 'ARI (classe primária)': True}
def compare_results(results):
    names = results.index.tolist()
    rows = []
    for metric in results.columns:
        a, b = results.loc[names[0], metric], results.loc[names[1], metric]
        winner = 'Empate' if np.isclose(a, b) else names[0] if ((a > b) == higher_is_better[metric]) else names[1]
        rows.append({'Métrica': metric, names[0]: a, names[1]: b,
                     'Melhor direção': 'Maior' if higher_is_better[metric] else 'Menor', 'Vencedor': winner})
    return pd.DataFrame(rows)
display(compare_results(direct_results).round(4))
plot_data = direct_results.reset_index().melt(id_vars='Método', var_name='Métrica', value_name='Valor')
fig = px.bar(plot_data, x='Método', y='Valor', color='Método', facet_col='Métrica',
             facet_col_wrap=3, title='Proximidade das classes originais no teste — K=54')
fig.update_yaxes(matches=None)
fig.update_layout(showlegend=False, height=650)
fig.show()

## Sonda linear: quanto dos rótulos pode ser recuperado?

Treinamos uma regressão logística por classe, com parâmetros iguais para os dois
métodos. Classes constantes no treino recebem uma previsão constante. Não excluímos
classes raras: avaliamos as 54 saídas e reportamos o suporte por classe.
O limiar 0,3 é pré-fixado; F1 depende desse limiar. LRAP dispensa essa escolha.

In [ ]:
def fit_probe(X_train, y_train, X_test):
    probabilities = np.zeros((len(X_test), y_train.shape[1]))
    for c in range(y_train.shape[1]):
        target = y_train[:, c]
        if len(np.unique(target)) == 1:
            probabilities[:, c] = target[0]
        else:
            clf = LogisticRegression(max_iter=500, C=1.0, random_state=SEED)
            clf.fit(X_train, target)
            probabilities[:, c] = clf.predict_proba(X_test)[:, 1]
    return probabilities

probe_rows, per_class_rows = [], []
for name, model in models.items():
    scores = fit_probe(model['train'], Y_train, model['test'])
    prediction = (scores >= THRESHOLD).astype(int)
    model['probe_scores'], model['prediction'] = scores, prediction
    metrics = ranking_metrics(Y_test, scores)
    metrics.update({
        'F1 micro': f1_score(Y_test, prediction, average='micro', zero_division=0),
        'F1 macro (54 classes)': f1_score(Y_test, prediction, average='macro', zero_division=0),
        'Jaccard por documento': jaccard_score(Y_test, prediction, average='samples', zero_division=0),
    })
    probe_rows.append({'Método': name, **metrics})
    precision, recall, f1, support = precision_recall_fscore_support(Y_test, prediction, average=None, zero_division=0)
    per_class_rows.append(pd.DataFrame({'Método': name, 'Classe': all_classes,
                                       'Precisão': precision, 'Recall': recall, 'F1': f1,
                                       'Suporte teste': support}))
probe_results = pd.DataFrame(probe_rows).set_index('Método')
per_class = pd.concat(per_class_rows, ignore_index=True)
higher_is_better.update({'F1 micro': True, 'F1 macro (54 classes)': True, 'Jaccard por documento': True})
display(compare_results(probe_results).round(4))
f1_by_class = per_class.pivot(index='Classe', columns='Método', values='F1')
f1_by_class['Diferença: híbrido - direto'] = f1_by_class['Híbrido (LLDA + LDA)'] - f1_by_class['LDA direto']
display(f1_by_class.join(class_support.set_index('Classe')[['Treino', 'Teste']]).sort_values('Diferença: híbrido - direto', ascending=False))
px.bar(per_class, x='Classe', y='F1', color='Método', barmode='group',
       hover_data=['Suporte teste', 'Precisão', 'Recall'],
       title='Recuperação das 54 classes pela sonda linear — teste', height=550).show()

## Heatmaps de alinhamento com todas as classes

Cada linha mostra a distribuição média dos 54 tópicos alinhados para os documentos
positivos de uma classe no teste. Uma diagonal concentrada favorece correspondência
direta; classes coocorrentes podem produzir blocos fora da diagonal. Classes ausentes
no teste aparecem sem valores. A escala é compartilhada entre os dois métodos.

In [ ]:
heatmaps = {}
for name, model in models.items():
    matrix = (Y_test.T @ model['aligned']) / np.maximum(Y_test.sum(0), 1)[:, None]
    matrix[Y_test.sum(0) == 0] = np.nan
    heatmaps[name] = matrix
vmax = max(np.nanmax(matrix) for matrix in heatmaps.values())
for name, matrix in heatmaps.items():
    px.imshow(matrix, x=all_classes, y=all_classes, zmin=0, zmax=vmax,
              labels={'x': 'Classe atribuída ao tópico no treino', 'y': 'Classe real no teste', 'color': 'Peso médio'},
              title=f'{name}: alinhamento tópico–classe (54 × 54)',
              color_continuous_scale='Blues', aspect='auto', height=850).show()

## Conclusão calculada e incerteza do resultado principal

Comparamos o LRAP alinhado por documento com bootstrap pareado no teste. O intervalo
é condicionado ao treino e à semente deste experimento: não mede variação entre
novos treinamentos. Se incluir zero, a evidência deste teste não distingue os métodos
pelo critério principal. Não somamos métricas com escalas diferentes nem decidimos
por contagem de vitórias.

In [ ]:
names = list(models)
per_doc_lrap = {
    name: np.array([label_ranking_average_precision_score(Y_test[i:i+1], model['aligned'][i:i+1])
                    for i in range(len(Y_test))])
    for name, model in models.items()
}
difference = per_doc_lrap[names[0]] - per_doc_lrap[names[1]]
rng = np.random.default_rng(SEED)
bootstrap = np.array([rng.choice(difference, size=len(difference), replace=True).mean() for _ in range(2000)])
low, high = np.quantile(bootstrap, [0.025, 0.975])
print(f'Diferença de LRAP alinhado (híbrido − direto): {difference.mean():+.4f}')
print(f'IC bootstrap pareado de 95%: [{low:+.4f}, {high:+.4f}]')
if low > 0:
    print('O híbrido chegou mais perto das classes originais pelo LRAP alinhado neste teste.')
elif high < 0:
    print('O LDA direto chegou mais perto das classes originais pelo LRAP alinhado neste teste.')
else:
    observed = direct_results['LRAP'].idxmax()
    print(f'Maior LRAP observado: {observed}; o intervalo inclui zero, sem diferença conclusiva.')
print(f"Melhor LRAP da sonda: {probe_results['LRAP'].idxmax()}.")
print(f"Melhor F1 macro da sonda: {probe_results['F1 macro (54 classes)'].idxmax()}.")
print('A sonda mede informação recuperável; o alinhamento mede correspondência direta.')
print('Interprete as classes raras pelo suporte e lembre que um único split/semente não estabelece superioridade geral.')

## Exportação opcional dos resultados

Execute a célula abaixo para salvar as tabelas, o split e os parâmetros em uma pasta
própria, sem sobrescrever os CSVs do experimento original.

In [ ]:
OUTPUT_DIR = DATA_DIR / 'resultados_comparativo_54_classes'
OUTPUT_DIR.mkdir(exist_ok=True)
direct_results.to_csv(OUTPUT_DIR / 'metricas_alinhamento.csv')
probe_results.to_csv(OUTPUT_DIR / 'metricas_sonda.csv')
per_class.to_csv(OUTPUT_DIR / 'metricas_por_classe.csv', index=False)
class_support.to_csv(OUTPUT_DIR / 'suporte_classes.csv', index=False)
pd.DataFrame({'doc_id': df['doc_id'], 'split': np.where(np.isin(np.arange(len(df)), train_idx), 'treino', 'teste')}).to_csv(OUTPUT_DIR / 'split.csv', index=False)
for name, model in models.items():
    slug = 'hibrido' if name == 'Híbrido (LLDA + LDA)' else 'lda_direto'
    model['alignment'].to_csv(OUTPUT_DIR / f'alinhamento_{slug}.csv', index=False)
import json
(OUTPUT_DIR / 'configuracao.json').write_text(json.dumps({
    'n_topics': N_TOPICS, 'seed': SEED, 'max_iter': MAX_ITER, 'test_size': TEST_SIZE,
    'threshold': THRESHOLD, 'classes': all_classes,
    'diferenca_lrap': float(difference.mean()), 'ic95': [float(low), float(high)],
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Resultados salvos em {OUTPUT_DIR}')